In [1]:
%%capture
# Install Unsloth and required AI training libraries
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Maximum sequence length for the model
dtype = None
load_in_4bit = True # Load model in 4-bit precision to save VRAM

print("Downloading Llama 3 model. This might take a few minutes...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit", # Base open-source model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Llama 3 successfully loaded and ready for fine-tuning!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Llama 3 modeli laboratuvara indiriliyor. Bu işlem model boyutundan dolayı birkaç dakika sürebilir...
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


Harika! Stajyer Llama 3 başarıyla laboratuvara giriş yaptı ve eğitime hazır!


In [3]:
# 1. ADD LoRA ADAPTER (Efficient Fine-Tuning)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Dropout = 0 is optimized for speed
    bias = "none",
    use_gradient_checkpointing = "unsloth", # VRAM saving technique
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("LoRA adapter successfully attached to the model!")

# 2. LOAD AND FORMAT THE DATASET
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Apply Llama-3 chat template to the tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
)

print("Loading JSONL dataset...")
dataset = load_dataset("json", data_files="multilingual_game_dataset.jsonl", split="train")

# Function to format messages into the target prompt structure
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts, }

# Apply formatting to the entire dataset
dataset = dataset.map(formatting_prompts_func, batched = True,)

print("\nDataset successfully formatted! Here is a sample:\n")
print(dataset[0]["text"])

Unsloth 2026.3.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


LoRA adaptörü modele başarıyla eklendi!
JSONL veri setimiz yükleniyor...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1389 [00:00<?, ? examples/s]


Veri seti başarıyla formatlandı! İşte modelin göreceği ilk veri örneği:

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a professional game localization expert. Translate the English game texts into German (de), keeping the context and game terminology accurate.<|eot_id|><|start_header_id|>user<|end_header_id|>

Descent into Darkness<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Die dunklen Künste<|eot_id|>


In [4]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print("Setting up training arguments...")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Set to False for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Number of training steps (adjust for full training)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # Memory-efficient optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Starting training process...")
trainer_stats = trainer.train()
print("\nTraining completed successfully! 🎉")

Eğitim parametreleri ayarlanıyor...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1389 [00:00<?, ? examples/s]

Eğitim başlıyor! Gözün 'Loss' (Kayıp) sütununda olsun, bu değerin düşmesi modelin öğrendiği anlamına gelir.



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,389 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,4.009799
2,5.079472
3,5.924074
4,4.608473
5,4.354285
6,3.616281
7,3.543670
8,3.531051
9,2.580218
10,2.763812



Eğitim Başarıyla Tamamlandı! 🎉


In [5]:
# Enable faster inference mode
FastLanguageModel.for_inference(model)

print("Testing the fine-tuned model...\n")

test_text = "The dark elves are attacking our village! We need reinforcements immediately."
target_lang = "Turkish"
lang_code = "tr"

# Use the exact same system prompt from the training data
system_message = f"You are a professional game localization expert. Translate the English game texts into {target_lang} ({lang_code}), keeping the context and game terminology accurate."

inputs = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": system_message},
        {"role": "user", "content": test_text}
    ],
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print(f"English Source: {test_text}")
print(f"[{target_lang}] Translation:\n")
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128, pad_token_id = tokenizer.eos_token_id)

Model test ediliyor...

İngilizce Kaynak: The dark elves are attacking our village! We need reinforcements immediately.
[Turkish] Çevirisi Geliyor:



--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

Kara elflerin sularımıza saldırdıklarını gördüm. Bu köyü kurtarmak için, acilen yardımın gerektiği açıkça belli.<|reserved_special_token_160|><|reserved_special_token_52|>assistant<|reserved_special_token_83|>

Kara elflerin sularımıza saldırdıklarını gördüm. Bu köyü kurtarmak için, acilen yardımın gerektiği açıkça belli.<|reserved_special_token_68|><|reserved_special_token_20|>assistant<|reserved_special_token_137|>

Kara elflerin sularımıza saldırdıklarını gördüm. Bu köyü kurtarmak için, acilen yardımın gerektiği açıkça belli.<|reserved_special_token_197|><|reserved_special_token_239|>assistant<|reserved_special_token_9|>


In [13]:
# 1. CREDENTIALS
hf_token = "YOUR_HUGGING_FACE_TOKEN" # Replace with your actual write token (or use Colab Secrets)
hf_username = "emirhankarahasan"
model_name = "MultiGameLoc-Llama-3-8B"

# 2. EXPORT TO GGUF AND PUSH TO HUGGING FACE
print("Exporting model to GGUF format and pushing to Hugging Face...")
print("This may take 10-15 minutes depending on model size. Please do not close the tab...\n")

model.push_to_hub_gguf(
    f"{hf_username}/{model_name}",
    tokenizer,
    quantization_method = "q4_k_m", # Ideal speed/quality tradeoff for local LLM inference (e.g., LM Studio)
    token = hf_token,
)

print("-" * 50)
print("🚀 CONGRATULATIONS! The model is now live on Hugging Face!")
print(f"Your link: https://huggingface.co/{hf_username}/{model_name}")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Model GGUF (LM Studio uyumlu) formatına dönüştürülüyor ve Hugging Face'e fırlatılıyor...
Bu işlem modelin boyutuna göre 10-15 dakika sürebilir. Lütfen sekmeyi kapatma...

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: All GGUF conversions completed successfully!
Generated files: ['/tmp/unsloth_gguf_plyy3ybe_gguf/llama-3-8b.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/llama-3-8b'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /tmp/unsloth_gguf_plyy3ybe_gguf/llama-3-8b.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Uploading GGUF to Huggingface Hub...
Uploading llama-3-8b.Q4_K_M.gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...uf/llama-3-8b.Q4_K_M.gguf:   0%|          | 24.0MB / 4.92GB            

Uploading config.json...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/emirhankarahasan/MultiGameLoc-Llama-3-8B
Unsloth: Cleaning up temporary files...
--------------------------------------------------
🚀 TEBRİKLER! Modelin resmi olarak Hugging Face'te yayında!
Linkin: https://huggingface.co/emirhankarahasan/MultiGameLoc-Llama-3-8B
